# DTU student data export

Fetches the student view from AIT's SQL Server, cleans it the same way as `update_static_data.ipynb`
cleans the old snapshot, and writes it to `static_data.students` in our PostgreSQL database, replacing
the previous run's table. The time goes to `static_data.fetched`, for the dashboard.

Configured entirely through environment variables (or a `.env` file), so it can run unattended.
The connections come from `get_ait_engine()` and `get_pg_engine()` in `pis_lib`:

| Variable | Required | Default |
|---|---|---|
| `AIT_USERNAME` | yes | |
| `AIT_PASSWORD` | yes | |
| `AIT_DATABASE_TABLE` | no | `View_DTUPythonSupport_GetStudentData` |
| `PGUSER` | no | the account running the notebook |
| `PGPASSWORD` | only off the server | |

The AIT server and database are set in `pis_lib/connect.py`. On the server, PostgreSQL needs
no password: it logs in as the account running the notebook.

The AIT connection checks the server's TLS certificate. A certificate error means the machine
doesn't trust the CA that signed it: install DTU's CA there.

Run headless with:

```bash
uv run --group student-data --with nbconvert jupyter nbconvert --to notebook --execute get_student_data.ipynb --output run.ipynb
```

In [ ]:
import logging
import os

import pandas as pd

from pis_lib.connect import get_ait_engine, get_pg_engine
from pis_lib.fetched import record_fetched

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
_logger = logging.getLogger("get_student_data")

### Configuration

In [ ]:
DATABASE_TABLE = os.getenv("AIT_DATABASE_TABLE", "View_DTUPythonSupport_GetStudentData")
PG_SCHEMA = "static_data"
PG_TABLE = "students"

In [ ]:
ait = get_ait_engine()
pg = get_pg_engine()

### Fetch data

In [ ]:
def fetch(query: str) -> pd.DataFrame:
    _logger.info("Executing: %s", query)
    with ait.connect() as conn:
        result = conn.exec_driver_sql(query)
        # object dtype keeps the values as the driver returns them: None stays None, and
        # integer columns with NULLs don't turn into floats
        return pd.DataFrame(result.fetchall(), columns=list(result.keys()), dtype=object)

In [ ]:
data = fetch(f"SELECT * FROM {DATABASE_TABLE}")
_logger.info("Fetched %d rows", len(data))

if data.empty:
    raise RuntimeError(f"Query returned no rows from {DATABASE_TABLE}")

### Clean

The same steps as the students section of `update_static_data.ipynb`, which loads the old snapshot:

1. **Rename** and **Reorder** the source columns to English names `COLUMN_NAMES` and in order `COLUMN_ORDER`.
2. **Gender**: map `M`/`K` to `Male`/`Female`; anything else becomes `Other`.
3. **Education**: normalise `uddannelse_dk` (lowercase, dots removed) and translate it with `STUDENT_EDU_MAP`. Rows with a missing or `-` code fall back to `EDU_FALLBACK`, looked up by direction. Anything still unmatched becomes `Other`.
4. **Direction**: collapse spelling variants and former programme names in `retning_dk` into one label each (`DIRECTION_ALIASES`).

In [ ]:
COLUMN_NAMES = {
    "KoenFraCpr"       : "gender",
    "stads_studentcode": "student_number",
    "retning_uk"       : "direction_EN",
    "retning_dk"       : "direction_DK",
    "uddannelse_dk"    : "education_DK",
}

COLUMN_ORDER = [
    "student_number",
    "gender",
    "education_EN",
    "education_DK",
    "direction_DK",
    "direction_EN",
]

GENDER_MAP = {"M": "Male", "K": "Female"}

  # keys are uddannelse_dk lowercased with dots removed ("ph.d" -> "phd", "åben udd." -> "åben udd")
STUDENT_EDU_MAP = {
    "bachelor"     : "BSc in Engineering",
    "kandidat"     : "MSc in Engineering",
    "diploming"    : "Diplom / BEng",
    "eksporting"   : "Diplom / BEng",
    "phd"          : "PhD",
    "deltidsdiplom": "Part-time diploma",
    "master"       : "Part-time master",
    "åben udd"     : "Open education",
    "enkeltfag"    : "Single course",
    "adgangskursus": "Access course",
    "gæst udl"     : "Exchange student",
    "gæst dk"      : "Guest student",
    "talent"       : "Talent programme",
}

  # used only for rows whose education code is "-" or missing
EDU_FALLBACK = {
    "Foreign exchange student": "Exchange student",
    "Deltidsdiplom"           : "Part-time diploma",
    "Tek. entreprenørskab"    : "MSc in Engineering",
}

# spelling variants in retning_dk -> one label per programme
DIRECTION_ALIASES = {
    "Medicin og tek"            : "Medicin og tek.",
    "Life science"              : "Life Science",
    "Miljøteknologi"            : "Miljøtek.",
    "Cyberteknologi"            : "Cybertek.",
    "Netværksteknologi"         : "Cybertek.",                          # former name
    "Mat. og tek."              : "Matematik og tek",
    "Data Science og Management": "Data science og management",
    "Bæredygtig energidesign"   : "Bæredygtigt energidesign",
    "Tek. entrepreneurskab"     : "Tek. entreprenørskab",
    "Farmateknologi"            : "Farmatek.",
    "Autonome systemer"         : "Autonome sys.",
    "Systembiologi"             : "Bioinfo. og systembio.",
    "Kemisk/biokemisk tek."     : "Kemisk og biokemisk tek.",
    "Biotek"                    : "Bioteknologi",
    "Industriel økonomi"        : "Industriel øko. og tekn. ledelse",
    "Mat. modellering"          : "Mat. model. og computing",
    "Konst. og mekanik"         : "Konstruktion og mek.",
    "Akustisk tek."             : "Lyd og akustisk tek.",
    "Telekom."                  : "Kom.tek. og system",
    "Digitale medietek."        : "Menneskeorienteret AI",              # former name
    "Transp. og logistik"       : "Transport og logistik",
    "Materialetek."             : "Materiale og procestek.",
    "Fødevareteknologi"         : "Fødevaretek.",
    "Maskin"                    : "Maskinteknik",
    "Kemi- og biotek."          : "Kemi/biotek.",
    "Vand, bioress. og miljø"   : "Miljøtek.",
    "Trafik og transport"       : "Mobilitet/transport",
    "IT elektronik"             : "Computer eng.",                      # retning_uk is "Computer Engineering"
    "Elektro"                   : "Elektrotek.",
}



students_df = (
    data
    .drop(columns=["displayName", "study_frame", "study_frame2", "uddannelse_uk"])
    .rename(columns=COLUMN_NAMES, errors="raise")
    .assign(
        gender=lambda d: d["gender"].map(GENDER_MAP).fillna("Other"),
        education_DK=lambda d: (
            d["education_DK"].str.lower().str.replace(".", "", regex=False).str.strip()
        ),
        education_EN=lambda d: (
            d["education_DK"].map(STUDENT_EDU_MAP)
            .fillna(d["direction_DK"].fillna(d["direction_EN"]).map(EDU_FALLBACK))
            .fillna("Other")
        ),
        direction_DK=lambda d: d["direction_DK"].str.strip().replace(DIRECTION_ALIASES),
    )
    [COLUMN_ORDER]
)

### Write to PostgreSQL

In [ ]:
with pg.begin() as conn:
    students_df.to_sql(PG_TABLE, conn, schema=PG_SCHEMA, if_exists="replace", index=False)
    record_fetched(conn, PG_TABLE)

_logger.info("Wrote %d rows to %s.%s", len(students_df), PG_SCHEMA, PG_TABLE)